In [1]:
import pandas as pd
import numpy as np

C:\Users\Vanathi\AppData\Roaming\Python\Python38\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Experiment Setup
Session-level randomized A/B test comparing generic failure messaging (Control) vs guided retry messaging (Treatment) after payment failures.

Assumptions:
1. Sessions are independent
2. Treatment effect applies only after a failure
3. Synthetic data reflects realistic payment behavior for demonstration purposes

# Load Data

In [2]:
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DATA_PATH = os.path.join(BASE_DIR, "data", "session_summary.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,session_id,variant,device,region,returning_user,initiated,failed,first_failure_type,succeeded,retry_count,time_to_success_sec
0,1,A,web,NaN,True,1,False,NaN,True,0,29.853190
1,2,B,web,APAC,False,1,False,NaN,True,0,42.930770
2,3,A,ios,NaN,True,1,False,NaN,True,0,48.890778
3,4,A,android,EU,True,1,False,NaN,True,0,52.216156
4,5,A,ios,APAC,True,1,False,NaN,True,0,29.704338


# Quick Sanity Check

In [4]:
# Variant split
df["variant"].value_counts(normalize=True)

variant
A    0.50092
B    0.49908
Name: proportion, dtype: float64

In [5]:
# Failure rate
df["failed"].mean()

0.14954

In [6]:
# Success rate overall
df["succeeded"].mean()

0.87808

# Primary Metric - Final Payment Success Rate

In [7]:
success_rate = (df.groupby("variant")["succeeded"].mean().rename("final_success_rate"))

success_rate


variant
A    0.870798
B    0.885389
Name: final_success_rate, dtype: float64

# Compute Uplift

In [8]:
uplift_abs = success_rate["B"] - success_rate["A"]
uplift_rel = uplift_abs / success_rate["A"]

uplift_abs, uplift_rel

(0.014591383800588997, 0.01675634106692123)

# Secondary Metric - Recovery Rate After Failure

In [9]:
failed_sessions = df[df["failed"] == True]

recovery_rate = (
    failed_sessions
        .groupby("variant")["succeeded"]
        .mean()
        .rename("recovery_rate")
)

recovery_rate

variant
A    0.133137
B    0.236111
Name: recovery_rate, dtype: float64

# Guardrails - Retries per session

In [10]:
df.groupby("variant")["retry_count"].mean()

variant
A    0.088996
B    0.137733
Name: retry_count, dtype: float64

# Time to success (successful sessions only)

In [11]:
df[df["succeeded"] == True].groupby("variant")["time_to_success_sec"].mean()

variant
A    45.012506
B    40.002916
Name: time_to_success_sec, dtype: float64

# Hard Decline rate 

In [12]:
hard_decline_rate = (df[df["failed"] == True].assign(is_hard = df["first_failure_type"] == "hard").groupby("variant")["is_hard"].mean()
)

hard_decline_rate


variant
A    0.296544
B    0.311699
Name: is_hard, dtype: float64

Results Summary

Treatment (B) shows higher final payment success and recovery rate after failure compared to control.

Retry count increased slightly in treatment but remained within acceptable bounds.

Time-to-success improved marginally.

Hard decline rates remained stable across variants.

Decision : Roll out guided retry messaging, with monitoring on retry frequency and time-to-success as guardrails.

In [13]:
from statsmodels.stats.proportion import proportions_ztest

# Success counts and totals
success_counts = df.groupby("variant")["succeeded"].sum()
total_counts = df.groupby("variant")["succeeded"].count()

count = [success_counts["A"], success_counts["B"]]
nobs = [total_counts["A"], total_counts["B"]]

z_stat, p_value = proportions_ztest(count, nobs)

z_stat, p_value

(-4.985930639265963, 6.166426523040129e-07)

The treatment shows a statistically significant improvement in final payment success rate (p < 0.001), making it unlikely the observed uplift is due to random chance.

Limitations

Data is synthetically generated to demonstrate experimentation logic rather than reproduce real payment logs.

User behavior is assumed to be independent across sessions.

Long-term behavioral effects and seasonality are not modeled.

Despite these limitations, the experiment demonstrates a robust, domain-agnostic approach to A/B testing and decision-making.